In [2]:
from train_model import train_model

старт модуля data_loader
старт модуля features
старт модуля preprocessing
старт модуля prepare_Data
Cтарт модуля modeling
Старт модуля train_model


"LightGBM"

In [3]:
y_pred, y_proba, results_baseline, x_train, x_test, y_train, y_test = train_model(name="LightGBM", name_experiment="baseline")

Accessing as Mu2Ka

Initialized MLflow to track repo "Mu2Ka/Credit-Default-Risk-mu2ka"

Repository Mu2Ka/Credit-Default-Risk-mu2ka initialized!

(246008, 381) (61503, 381)
Получили датасеты
Начинаем обучение моделей
Now we start training LightGBM
Model: LightGBM, Accuracy: 0.9201014584654407, ROC AUC: 0.7829709056465405, PR AUC: 0.2795429210968179


2026/11/08 15:37:56 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
Registered model 'credit_default_risk_baseline_model' already exists. Creating a new version of this model...
2026/11/08 15:38:11 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: credit_default_risk_baseline_model, version 8
Created version '8' of model 'credit_default_risk_baseline_model'.


🏃 View run baseline_LightGBM at: https://dagshub.com/Mu2Ka/Credit-Default-Risk-mu2ka.mlflow/#/experiments/0/runs/98429690ff014b158f2ab9a35e711755
🧪 View experiment at: https://dagshub.com/Mu2Ka/Credit-Default-Risk-mu2ka.mlflow/#/experiments/0


In [11]:
import pandas as pd

In [12]:
y_pred = pd.Series(y_pred, name="y_pred",index = y_test.index)

In [18]:
y_proba = pd.Series(y_proba, name="y_proba",index = y_test.index)

In [14]:
y_pred

256571    0
191493    0
103497    0
130646    0
211898    0
         ..
16213     0
294620    0
234384    0
149027    0
268861    0
Name: y_pred, Length: 61503, dtype: int64

In [13]:
y_test

256571    0
191493    0
103497    0
130646    0
211898    0
         ..
16213     0
294620    0
234384    0
149027    1
268861    0
Name: TARGET, Length: 61503, dtype: int64

In [20]:
x_test['result'] = y_test - y_pred   #1 - FN, 0 - TN,-1 - FP
x_test['result'].value_counts()
x_test['proba'] = y_proba

C:\Users\myska\AppData\Local\Temp\ipykernel_26052\2242009161.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  x_test['proba'] = y_proba


In [25]:

avg_confidence_fn = (1 - x_test.loc[x_test["result"] == 1, "proba"]).mean()

avg_confidence_fp = x_test.loc[x_test["result"] == -1, "proba"].mean()

tn = (x_test["result"] == 0) & (y_test == 0)
avg_confidence_tn = (1 - x_test.loc[tn, "proba"]).mean()

In [26]:
print(f"Average confidence for false negatives: {avg_confidence_fn:.4f}")
print(f"Average confidence for false positives: {avg_confidence_fp:.4f}")
print(f"Average confidence for true negatives: {avg_confidence_tn:.4f}")

Average confidence for false negatives: 0.8368
Average confidence for false positives: 0.5536
Average confidence for true negatives: 0.9295


In [27]:
analysis = x_test.copy()
analysis = x_test.copy()
analysis["actual"] = y_test
analysis["pred"] = pd.Series(y_pred, index=y_test.index)
analysis["proba"] = pd.Series(y_proba, index=y_test.index)

analysis["error"] = analysis["actual"] != analysis["pred"]
analysis["fn"] = (analysis["actual"] == 1) & (analysis["pred"] == 0)
analysis["fp"] = (analysis["actual"] == 0) & (analysis["pred"] == 1)

In [30]:
analysis['actual'].value_counts()[0] + analysis['actual'].value_counts()[1] == 61503

np.True_

In [32]:
analysis['proba_group'] = pd.qcut(analysis['proba'], q=10, duplicates='drop')
analysis['proba_group']
analysis["credit_group"] = pd.qcut(analysis["AMT_CREDIT"], 5, duplicates="drop")
analysis["income_group"] = pd.qcut(analysis["AMT_INCOME_TOTAL"], 5, duplicates="drop")

In [34]:
def grouped_analysis(column):
    return analysis.groupby(column).agg(
        clients=('actual', 'size'),
        default_rate=("actual", "mean"),
        mean_proba=("proba", "mean"),
        errors=("error", "sum"),
        false_negatives=("fn", "sum"),
        false_positives=("fp", "sum"),
    )   
    
grouped_analysis("proba_group")


,clients,default_rate,mean_proba,errors,false_negatives,false_positives
proba_group,,,,,,
"(0.00234, 0.0157]",6151,0.010242,0.011915,63,63,0
"(0.0157, 0.022]",6150,0.018211,0.018860,112,112,0
"(0.022, 0.029]",6150,0.023415,0.025416,144,144,0
"(0.029, 0.0375]",6150,0.027967,0.033046,172,172,0
"(0.0375, 0.0487]",6151,0.042432,0.042840,261,261,0
"(0.0487, 0.0632]",6150,0.053496,0.055562,329,329,0
"(0.0632, 0.0844]",6150,0.070732,0.073052,435,435,0
"(0.0844, 0.119]",6150,0.105203,0.100233,647,647,0
"(0.119, 0.187]",6150,0.156748,0.148460,964,964,0


In [35]:
grouped_analysis("credit_group")

,clients,default_rate,mean_proba,errors,false_negatives,false_positives
credit_group,,,,,,
"(44999.999, 254700.0]",13024,0.071483,0.073306,925,911,14
"(254700.0, 427500.0]",11596,0.090721,0.090448,1051,1015,36
"(427500.0, 604152.0]",12350,0.100243,0.099081,1212,1166,46
"(604152.0, 900000.0]",12797,0.080331,0.077562,1010,996,14
"(900000.0, 4050000.0]",11736,0.061009,0.059674,716,710,6


In [36]:
grouped_analysis("income_group")

,clients,default_rate,mean_proba,errors,false_negatives,false_positives
income_group,,,,,,
"(25649.999, 99000.0]",12698,0.079304,0.082339,1006,977,29
"(99000.0, 135000.0]",17169,0.086085,0.085664,1452,1423,29
"(135000.0, 166500.0]",7377,0.089332,0.084301,648,632,16
"(166500.0, 225000.0]",14771,0.079954,0.077367,1172,1142,30
"(225000.0, 13500000.0]",9488,0.067454,0.067363,636,624,12


In [39]:
from sklearn.metrics import roc_auc_score

print("AUC scores by credit group:")

for group,part in analysis.groupby("credit_group",observed = True):
    if part["actual"].nunique() == 2:
        auc = roc_auc_score(part["actual"], part["proba"])
        print(f"Group: {group}, AUC: {auc:.4f}")
print("AUC scores by income group:")
for group,part in analysis.groupby("income_group",observed = True):
    if part["actual"].nunique() == 2:
        auc = roc_auc_score(part["actual"], part["proba"])
        print(f"Group: {group}, AUC: {auc:.4f}")
        
        

AUC scores by credit group:
Group: (44999.999, 254700.0], AUC: 0.7780
Group: (254700.0, 427500.0], AUC: 0.7915
Group: (427500.0, 604152.0], AUC: 0.7905
Group: (604152.0, 900000.0], AUC: 0.7688
Group: (900000.0, 4050000.0], AUC: 0.7629
AUC scores by income group:
Group: (25649.999, 99000.0], AUC: 0.7799
Group: (99000.0, 135000.0], AUC: 0.7792
Group: (135000.0, 166500.0], AUC: 0.7808
Group: (166500.0, 225000.0], AUC: 0.7774
Group: (225000.0, 13500000.0], AUC: 0.8028


In [47]:
analysis["ratio_group"] = pd.qcut(
    analysis["credit_income_credit"], 5, duplicates="drop"
)

for group, part in analysis.groupby("ratio_group", observed=True):
    if part["actual"].nunique() == 2:
        print(group, round(roc_auc_score(part["actual"], part["proba"]), 4))
        
part['high_group'] = pd.qcut(part['proba'], q=10, duplicates='drop')
part_high = part
part_high.groupby("high_group", observed=True).agg(
    clients=("actual", "size"),
    default_rate=("actual", "mean"),
    mean_proba=("proba", "mean"),
)

(0.103, 1.818] 0.785
(1.818, 2.764] 0.7836
(2.764, 3.894] 0.7962
(3.894, 5.752] 0.7838
(5.752, 84.737] 0.7587


,clients,default_rate,mean_proba
high_group,,,
"(0.0030800000000000003, 0.0166]",1230,0.018699,0.012733
"(0.0166, 0.022]",1230,0.021951,0.019295
"(0.022, 0.0283]",1230,0.025203,0.025145
"(0.0283, 0.0358]",1230,0.017886,0.031860
"(0.0358, 0.0452]",1230,0.046341,0.040354
"(0.0452, 0.0575]",1229,0.050448,0.050992
"(0.0575, 0.0734]",1230,0.057724,0.065087
"(0.0734, 0.102]",1230,0.092683,0.086328
"(0.102, 0.16]",1230,0.141463,0.127651


In [41]:
analysis.groupby("ratio_group", observed=True).agg(
    clients=("actual", "size"),
    default_rate=("actual", "mean"),
    mean_proba=("proba", "mean"),
)

,clients,default_rate,mean_proba
ratio_group,,,
"(0.103, 1.818]",12308,0.074098,0.074581
"(1.818, 2.764]",12293,0.083788,0.084109
"(2.764, 3.894]",12302,0.089823,0.086552
"(3.894, 5.752]",12301,0.083814,0.083368
"(5.752, 84.737]",12299,0.072120,0.071383
